# Yuanta Brokerage Data — Exploratory Data Analysis (EDA)

**Date:** 2026-03-12  
**Data file date:** 2026-03-09  
**Purpose:** Identify data quality issues, anomalies, and define cleaning rules before building the pipeline.

### Files inspected
| File | Description |
|------|-------------|
| `input/clients.csv` | Client master / reference data |
| `input/instruments.csv` | Instrument master / reference data |
| `input/trades_2026-03-09.csv` | Daily trade events |

In [2]:
import pandas as pd
import numpy as np

# Paths
BASE = "yuanta-data-engineer-assessment-test/input"
CLIENTS_PATH    = f"{BASE}/clients.csv"
INSTRUMENTS_PATH = f"{BASE}/instruments.csv"
TRADES_PATH     = f"{BASE}/trades_2026-03-09.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 120)

print("Libraries loaded.")

Libraries loaded.


---
## 1. `clients.csv`

In [3]:
clients = pd.read_csv(CLIENTS_PATH)
print(f"Shape: {clients.shape}")
clients

Shape: (15, 5)


,client_id,client_name,country,kyc_status,created_at
0,C001,Alice Capital,US,APPROVED,2024-01-15
1,C002,Bob Investments,SG,APPROVED,2024-03-22
2,C003,Charlie Partners,GB,PENDING,2025-06-10
3,C004,Delta Trading,ID,REJECTED,2023-11-05
4,C005,Echo Fund,NaN,APPROVED,2025-12-01
5,C006,Foxtrot Securities,US,APPROVED,2022-08-19
6,C007,Gamma Holdings,AU,APPROVED,2023-02-14
7,C008,Helios Asset Mgmt,DE,PENDING,2025-01-09
8,C009,Indigo Capital,JP,APPROVED,2021-05-30
9,C010,Jade Partners,HK,REJECTED,2024-09-12


In [4]:
print("=== dtypes ===")
print(clients.dtypes)
print("\n=== Null counts ===")
print(clients.isnull().sum())
print("\n=== kyc_status distribution ===")
print(clients["kyc_status"].value_counts())

=== dtypes ===
client_id      object
client_name    object
country        object
kyc_status     object
created_at     object
dtype: object

=== Null counts ===
client_id      0
client_name    0
country        1
kyc_status     0
created_at     0
dtype: int64

=== kyc_status distribution ===
kyc_status
APPROVED    10
PENDING      3
REJECTED     2
Name: count, dtype: int64


In [5]:
issues_clients = []

# 1. Duplicate client_id
dup_ids = clients[clients.duplicated("client_id", keep=False)]
if not dup_ids.empty:
    issues_clients.append(("DUPLICATE client_id", dup_ids))

# 2. Missing values
missing = clients[clients.isnull().any(axis=1)]
if not missing.empty:
    issues_clients.append(("MISSING VALUES", missing))

# 3. Non-APPROVED kyc_status (flag for business rule)
non_approved = clients[clients["kyc_status"] != "APPROVED"]
if not non_approved.empty:
    issues_clients.append(("NON-APPROVED KYC (flag)", non_approved))

for label, df in issues_clients:
    print(f"\n{'='*50}\n[ISSUE] {label}\n{'='*50}")
    print(df.to_string(index=False))


[ISSUE] MISSING VALUES
client_id client_name country kyc_status created_at
     C005   Echo Fund     NaN   APPROVED 2025-12-01

[ISSUE] NON-APPROVED KYC (flag)
client_id       client_name country kyc_status created_at
     C003  Charlie Partners      GB    PENDING 2025-06-10
     C004     Delta Trading      ID   REJECTED 2023-11-05
     C008 Helios Asset Mgmt      DE    PENDING 2025-01-09
     C010     Jade Partners      HK   REJECTED 2024-09-12
     C013 Mango Investments      ID    PENDING 2025-10-02


---
## 2. `instruments.csv`

In [6]:
instruments = pd.read_csv(INSTRUMENTS_PATH)
print(f"Shape: {instruments.shape}")
instruments

Shape: (20, 5)


,instrument_id,symbol,asset_class,currency,exchange
0,I001,AAPL,EQUITY,USD,NASDAQ
1,I002,TSLA,EQUITY,USD,NASDAQ
2,I003,EURUSD,FX,USD,OTC
3,I004,BBCA,EQUITY,IDR,IDX
4,I005,BTCUSD,CRYPTO,USD,CRYPTOX
5,I006,MSFT,EQUITY,USD,NASDAQ
6,I007,GOOGL,EQUITY,USD,NASDAQ
7,I008,AMZN,EQUITY,USD,NASDAQ
8,I009,USDJPY,FX,JPY,OTC
9,I010,XAUUSD,COMMODITY,USD,OTC


In [7]:
print("=== dtypes ===")
print(instruments.dtypes)
print("\n=== Null counts ===")
print(instruments.isnull().sum())
print("\n=== asset_class distribution ===")
print(instruments["asset_class"].value_counts())
print("\n=== Duplicate instrument_id ===")
print(instruments.duplicated("instrument_id").sum())

=== dtypes ===
instrument_id    object
symbol           object
asset_class      object
currency         object
exchange         object
dtype: object

=== Null counts ===
instrument_id    0
symbol           0
asset_class      0
currency         0
exchange         0
dtype: int64

=== asset_class distribution ===
asset_class
EQUITY       11
FX            3
CRYPTO        2
ETF           2
COMMODITY     1
BOND          1
Name: count, dtype: int64

=== Duplicate instrument_id ===
0


---
## 3. `trades_2026-03-09.csv`

In [8]:
# Load trades keeping all columns as string first so we can detect formatting issues
trades_raw = pd.read_csv(TRADES_PATH, dtype=str)
print(f"Shape (raw): {trades_raw.shape}")
trades_raw

Shape (raw): (50, 9)


,trade_id,trade_time,client_id,instrument_id,side,quantity,price,fees,status
0,T0001,2026-03-09T09:15:00Z,C001,I001,BUY,10,185.25,1.25,EXECUTED
1,T0002,2026-03-09T09:16:10Z,C002,I002,SELL,5,250.10,0.90,EXECUTED
2,T0003,2026-03-09T09:17:30Z,C999,I001,BUY,2,185.25,0.25,EXECUTED
3,T0004,2026-03-09T09:18:05Z,C003,I004,BUY,1000,"1,950",10.00,EXECUTED
4,T0005,2026-03-09T09:19:00Z,C003,I005,BUY,0,82000,15.00,EXECUTED
5,T0006,2026-03-09T09:20:00Z,C001,I002,HOLD,1,250.10,0.10,EXECUTED
6,T0007,2026-03-09T09:21:00Z,C004,I001,SELL,3,-10,0.10,EXECUTED
7,T0008,2026-03-09T09:22:00Z,C005,I003,BUY,10000,1.0850,NaN,CANCELLED
8,T0002,2026-03-09T09:25:10Z,C002,I002,SELL,5,250.10,0.90,EXECUTED
9,T0009,2026-03-09T09:23:00Z,C001,I001,buy,7,185.25,0.70,EXECUTED


In [9]:
print("=== dtypes ===")
print(trades_raw.dtypes)
print("\n=== Null counts ===")
print(trades_raw.isnull().sum())
print("\n=== status distribution ===")
print(trades_raw["status"].value_counts())
print("\n=== side distribution (raw, before cleaning) ===")
print(trades_raw["side"].value_counts())

=== dtypes ===
trade_id         object
trade_time       object
client_id        object
instrument_id    object
side             object
quantity         object
price            object
fees             object
status           object
dtype: object

=== Null counts ===
trade_id         0
trade_time       0
client_id        0
instrument_id    0
side             0
quantity         0
price            1
fees             1
status           0
dtype: int64

=== status distribution ===
status
EXECUTED     48
CANCELLED     2
Name: count, dtype: int64

=== side distribution (raw, before cleaning) ===
side
BUY      30
SELL     18
HOLD      1
 buy      1
Name: count, dtype: int64


### 3.1 Duplicate `trade_id`

In [10]:
dup_trades = trades_raw[trades_raw.duplicated("trade_id", keep=False)].sort_values("trade_id")
print(f"Rows with duplicate trade_id: {len(dup_trades)}")
print()
print(dup_trades.to_string(index=False))

print("\n--- Classification ---")
for tid, grp in dup_trades.groupby("trade_id"):
    # exact duplicate = all fields identical except trade_time
    cols_excl_time = [c for c in grp.columns if c != "trade_time"]
    is_exact = grp[cols_excl_time].duplicated(keep=False).all()
    kind = "EXACT DUPLICATE" if is_exact else "LATE UPDATE / AMENDMENT"
    print(f"  {tid}: {kind}")

Rows with duplicate trade_id: 4

trade_id           trade_time client_id instrument_id side quantity  price fees   status
   T0002 2026-03-09T09:16:10Z      C002          I002 SELL        5 250.10 0.90 EXECUTED
   T0002 2026-03-09T09:25:10Z      C002          I002 SELL        5 250.10 0.90 EXECUTED
   T0034 2026-03-09T09:47:00Z      C001          I001  BUY       10 185.25 1.25 EXECUTED
   T0034 2026-03-09T09:48:00Z      C001          I001  BUY       10 185.30 1.25 EXECUTED

--- Classification ---
  T0002: EXACT DUPLICATE
  T0034: LATE UPDATE / AMENDMENT


### 3.2 Invalid Foreign Keys (client_id / instrument_id)

In [11]:
valid_client_ids     = set(clients["client_id"].str.strip())
valid_instrument_ids = set(instruments["instrument_id"].str.strip())

# strip whitespace from raw trades for the FK check
trades_raw["client_id_clean"]     = trades_raw["client_id"].str.strip()
trades_raw["instrument_id_clean"] = trades_raw["instrument_id"].str.strip()

invalid_client_fk = trades_raw[~trades_raw["client_id_clean"].isin(valid_client_ids)]
invalid_instr_fk  = trades_raw[~trades_raw["instrument_id_clean"].isin(valid_instrument_ids)]

print(f"[FK] Trades with unknown client_id: {len(invalid_client_fk)}")
print(invalid_client_fk[["trade_id","trade_time","client_id","instrument_id","side","status"]].to_string(index=False))

print(f"\n[FK] Trades with unknown instrument_id: {len(invalid_instr_fk)}")
print(invalid_instr_fk[["trade_id","trade_time","client_id","instrument_id","side","status"]].to_string(index=False))

[FK] Trades with unknown client_id: 1
trade_id           trade_time client_id instrument_id side   status
   T0003 2026-03-09T09:17:30Z      C999          I001  BUY EXECUTED

[FK] Trades with unknown instrument_id: 1
trade_id           trade_time client_id instrument_id side   status
   T0025 2026-03-09T09:38:00Z      C007          I999 SELL EXECUTED


### 3.3 Whitespace & Casing Issues

In [12]:
VALID_SIDES = {"BUY", "SELL"}

# Rows where client_id or side differ after strip+upper
whitespace_client = trades_raw[trades_raw["client_id"] != trades_raw["client_id"].str.strip()]
casing_side       = trades_raw[trades_raw["side"].str.strip().str.upper() != trades_raw["side"]]

print(f"[WHITESPACE] client_id has leading/trailing spaces: {len(whitespace_client)}")
print(whitespace_client[["trade_id","client_id","side"]].to_string(index=False))

print(f"\n[CASING/WHITESPACE] side is not normalised: {len(casing_side)}")
print(casing_side[["trade_id","client_id","side"]].to_string(index=False))

# Invalid side values (after normalisation)
invalid_side = trades_raw[~trades_raw["side"].str.strip().str.upper().isin(VALID_SIDES)]
print(f"\n[INVALID SIDE] values outside BUY/SELL: {len(invalid_side)}")
print(invalid_side[["trade_id","client_id","side","status"]].to_string(index=False))

[WHITESPACE] client_id has leading/trailing spaces: 1
trade_id client_id  side
   T0009     C001   buy 

[CASING/WHITESPACE] side is not normalised: 1
trade_id client_id  side
   T0009     C001   buy 

[INVALID SIDE] values outside BUY/SELL: 1
trade_id client_id side   status
   T0006      C001 HOLD EXECUTED


### 3.4 Numeric Formatting Issues (comma-separated numbers as strings)

In [13]:
def has_comma_in_number(series):
    """Return boolean mask: string contains a digit-comma-digit pattern."""
    return series.fillna("").str.contains(r'\d,\d', regex=True)

comma_price    = trades_raw[has_comma_in_number(trades_raw["price"])]
comma_quantity = trades_raw[has_comma_in_number(trades_raw["quantity"])]
comma_fees     = trades_raw[has_comma_in_number(trades_raw["fees"])]

print(f"[COMMA IN PRICE]    {len(comma_price)} rows")
print(comma_price[["trade_id","quantity","price","fees"]].to_string(index=False))

print(f"\n[COMMA IN QUANTITY] {len(comma_quantity)} rows")
print(comma_quantity[["trade_id","quantity","price","fees"]].to_string(index=False))

print(f"\n[COMMA IN FEES]     {len(comma_fees)} rows")
print(comma_fees[["trade_id","quantity","price","fees"]].to_string(index=False))

[COMMA IN PRICE]    2 rows
trade_id quantity    price  fees
   T0004     1000    1,950 10.00
   T0032        1 3,100.00  3.00

[COMMA IN QUANTITY] 1 rows
trade_id quantity price fees
   T0012    1,000 98.75 5.00

[COMMA IN FEES]     0 rows
Empty DataFrame
Columns: [trade_id, quantity, price, fees]
Index: []


### 3.5 Missing, Zero, and Negative Numeric Values

In [14]:
# Parse numerics (strip commas first)
def to_numeric(series):
    return pd.to_numeric(series.str.replace(",", "", regex=False), errors="coerce")

trades_num = trades_raw.copy()
trades_num["quantity_num"] = to_numeric(trades_num["quantity"])
trades_num["price_num"]    = to_numeric(trades_num["price"])
trades_num["fees_num"]     = to_numeric(trades_num["fees"])

cols_display = ["trade_id","trade_time","client_id","instrument_id","quantity","price","fees"]

# Missing price
missing_price = trades_num[trades_num["price_num"].isna()]
print(f"[MISSING PRICE] {len(missing_price)} rows")
print(missing_price[cols_display].to_string(index=False))

# Missing quantity
missing_qty = trades_num[trades_num["quantity_num"].isna()]
print(f"\n[MISSING QUANTITY] {len(missing_qty)} rows")
print(missing_qty[cols_display].to_string(index=False))

# Missing fees
missing_fees = trades_num[trades_num["fees_num"].isna()]
print(f"\n[MISSING FEES] {len(missing_fees)} rows")
print(missing_fees[cols_display + ["status"]].to_string(index=False))

# Zero quantity
zero_qty = trades_num[trades_num["quantity_num"] == 0]
print(f"\n[ZERO QUANTITY] {len(zero_qty)} rows")
print(zero_qty[cols_display].to_string(index=False))

# Negative price
neg_price = trades_num[trades_num["price_num"] < 0]
print(f"\n[NEGATIVE PRICE] {len(neg_price)} rows")
print(neg_price[cols_display].to_string(index=False))

[MISSING PRICE] 1 rows
trade_id           trade_time client_id instrument_id quantity price fees
   T0030 2026-03-09T09:43:00Z      C012          I004      100   NaN 0.25

[MISSING QUANTITY] 0 rows
Empty DataFrame
Columns: [trade_id, trade_time, client_id, instrument_id, quantity, price, fees]
Index: []

[MISSING FEES] 1 rows
trade_id           trade_time client_id instrument_id quantity  price fees    status
   T0008 2026-03-09T09:22:00Z      C005          I003    10000 1.0850  NaN CANCELLED

[ZERO QUANTITY] 2 rows
trade_id           trade_time client_id instrument_id quantity  price  fees
   T0005 2026-03-09T09:19:00Z      C003          I005        0  82000 15.00
   T0026 2026-03-09T09:39:00Z      C008          I006        0 410.55  0.10

[NEGATIVE PRICE] 1 rows
trade_id           trade_time client_id instrument_id quantity price fees
   T0007 2026-03-09T09:21:00Z      C004          I001        3   -10 0.10


### 3.6 Trades from Clients with Non-APPROVED KYC

In [15]:
non_approved_clients = clients[clients["kyc_status"] != "APPROVED"][["client_id","client_name","kyc_status"]]
print("Non-APPROVED clients:")
print(non_approved_clients.to_string(index=False))

kyc_trades = (
    trades_raw
    .merge(non_approved_clients, left_on="client_id_clean", right_on="client_id", how="inner")
    .rename(columns={"client_id_x": "client_id_raw"})
)

print(f"\nTrades from non-APPROVED clients: {len(kyc_trades)}")
print(kyc_trades[["trade_id","trade_time","client_id_raw","client_name","kyc_status","instrument_id","side","status"]].to_string(index=False))

Non-APPROVED clients:
client_id       client_name kyc_status
     C003  Charlie Partners    PENDING
     C004     Delta Trading   REJECTED
     C008 Helios Asset Mgmt    PENDING
     C010     Jade Partners   REJECTED
     C013 Mango Investments    PENDING

Trades from non-APPROVED clients: 16
trade_id           trade_time client_id_raw       client_name kyc_status instrument_id side    status
   T0004 2026-03-09T09:18:05Z          C003  Charlie Partners    PENDING          I004  BUY  EXECUTED
   T0005 2026-03-09T09:19:00Z          C003  Charlie Partners    PENDING          I005  BUY  EXECUTED
   T0007 2026-03-09T09:21:00Z          C004     Delta Trading   REJECTED          I001 SELL  EXECUTED
   T0012 2026-03-09T09:28:00Z          C008 Helios Asset Mgmt    PENDING          I008  BUY  EXECUTED
   T0014 2026-03-09T09:29:05Z          C010     Jade Partners   REJECTED          I010 SELL  EXECUTED
   T0017 2026-03-09T09:31:30Z          C013 Mango Investments    PENDING          I013 SELL  E

---
## 4. Anomaly Summary & Cleaning Rules

| # | Table | Anomaly | Affected Rows | Cleaning Rule |
|---|-------|---------|--------------|---------------|
| 1 | `clients` | Missing `country` | C005 | Load as NULL; flag for enrichment |
| 2 | `clients` | Non-APPROVED KYC clients trading | C003 (PENDING), C004/C010 (REJECTED) | Load with `kyc_flag` column; business decision |
| 3 | `trades` | Exact duplicate `trade_id` (T0002) | 1 extra row | Keep latest `trade_time`, drop earlier |
| 4 | `trades` | Late-update / amended `trade_id` (T0034) | 1 pair | Keep latest `trade_time` row as canonical |
| 5 | `trades` | Invalid FK `client_id = C999` | T0003 | Quarantine to rejected table |
| 6 | `trades` | Invalid FK `instrument_id = I999` | T0025 | Quarantine to rejected table |
| 7 | `trades` | Whitespace + lowercase in `client_id` / `side` | T0009 | `TRIM()` + `UPPER()` before FK lookup |
| 8 | `trades` | Invalid `side = "HOLD"` | T0006 | Quarantine to rejected table |
| 9 | `trades` | Negative `price` | T0007 | Quarantine to rejected table |
| 10 | `trades` | Zero `quantity` | T0005, T0026 | Quarantine to rejected table |
| 11 | `trades` | Missing `price` | T0030 | Quarantine to rejected table |
| 12 | `trades` | Missing `fees` | T0008 | Default to 0.0 if CANCELLED, else quarantine |
| 13 | `trades` | Price with comma separator `"1,950"` | T0004, T0032 | Strip commas, cast to FLOAT |
| 14 | `trades` | Quantity with comma separator `"1,000"` | T0012 | Strip commas, cast to INT/FLOAT |

In [16]:
print("=" * 60)
print("FINAL ANOMALY COUNT SUMMARY")
print("=" * 60)

QUARANTINE_IDS = {"T0003", "T0006", "T0007", "T0025", "T0026", "T0030"}

summary = {
    "Total raw trade rows":            len(trades_raw),
    "Exact duplicate rows (drop)":     1,   # T0002 second occurrence
    "Late-update pairs (keep latest)": 1,   # T0034
    "Invalid FK - client":             1,   # C999
    "Invalid FK - instrument":         1,   # I999
    "Invalid side value (HOLD)":       1,   # T0006
    "Negative price":                  1,   # T0007
    "Zero quantity":                   2,   # T0005, T0026
    "Missing price":                   1,   # T0030
    "Missing fees (CANCELLED)":        1,   # T0008
    "Comma-formatted numerics":        3,   # T0004 price, T0012 qty, T0032 price
    "Whitespace/casing issues":        1,   # T0009
    "Estimated clean & loadable rows": "~39–41",
}

for k, v in summary.items():
    print(f"  {k:<40} {v}")

FINAL ANOMALY COUNT SUMMARY
  Total raw trade rows                     50
  Exact duplicate rows (drop)              1
  Late-update pairs (keep latest)          1
  Invalid FK - client                      1
  Invalid FK - instrument                  1
  Invalid side value (HOLD)                1
  Negative price                           1
  Zero quantity                            2
  Missing price                            1
  Missing fees (CANCELLED)                 1
  Comma-formatted numerics                 3
  Whitespace/casing issues                 1
  Estimated clean & loadable rows          ~39–41
